# Bronze — Ingesta y consolidación de datos raw
**Proyecto:** AI Tech Landscape Pipeline  
**Autor:** Carlos Díaz  
**Descripción:** Lee todos los CSV desde `/data/bronze/`, los consolida por fuente y los guarda como Parquet en `/data/silver/`.


In [ ]:
import pandas as pd
import os
import glob

# ── Rutas ────────────────────────────────────────────────────
BRONZE_PATH = "../data/bronze/"
SILVER_PATH = "../data/silver/"
os.makedirs(SILVER_PATH, exist_ok=True)
print("✓ Rutas configuradas")

## Bloque 1 — Big Tech Stock Prices
`archive/` — 14 empresas: AAPL, ADBE, AMZN, CRM, CSCO, GOOGL, IBM, INTC, META, MSFT, NFLX, NVDA, ORCL, TSLA

In [ ]:
archivos_big_tech = glob.glob(os.path.join(BRONZE_PATH, "archive", "*.csv"))
print(f"Archivos encontrados: {len(archivos_big_tech)}")

frames_big_tech = []
for filepath in archivos_big_tech:
    ticker = os.path.basename(filepath).replace(".csv", "")
    df = pd.read_csv(filepath)
    df["ticker"] = ticker
    df["source_file"] = os.path.basename(filepath)
    frames_big_tech.append(df)
    print(f"  ✓ {ticker:6s} — {len(df):,} filas")

df_big_tech = pd.concat(frames_big_tech, ignore_index=True)
print(f"\nTotal consolidado: {len(df_big_tech):,} filas")
print(f"Columnas: {list(df_big_tech.columns)}")
df_big_tech.head(3)

In [ ]:
df_big_tech.to_parquet(f"{SILVER_PATH}raw_big_tech_stocks.parquet", index=False)
print("✓ Guardado: silver/raw_big_tech_stocks.parquet")

## Bloque 2 — GPU Companies
`archive (1)/` — AMD, ASUS, INTEL, NVIDIA, MSI

In [ ]:
archivos_gpu = glob.glob(os.path.join(BRONZE_PATH, "archive (1)", "*.csv"))
print(f"Archivos encontrados: {len(archivos_gpu)}")

frames_gpu = []
for filepath in archivos_gpu:
    nombre = os.path.basename(filepath)
    empresa = nombre.split(" ")[0].upper()
    df = pd.read_csv(filepath)
    df["empresa"] = empresa
    df["source_file"] = nombre
    frames_gpu.append(df)
    print(f"  ✓ {empresa:8s} — {len(df):,} filas")

df_gpu = pd.concat(frames_gpu, ignore_index=True)
print(f"\nTotal consolidado: {len(df_gpu):,} filas")
print(f"Columnas: {list(df_gpu.columns)}")
df_gpu.head(3)

In [ ]:
df_gpu.to_parquet(f"{SILVER_PATH}raw_gpu_companies.parquet", index=False)
print("✓ Guardado: silver/raw_gpu_companies.parquet")

## Bloque 3 — AI Companies
`archive (2)/Ai_companies.csv` — métricas de negocio

In [ ]:
df_ai = pd.read_csv(os.path.join(BRONZE_PATH, "archive (2)", "Ai_companies.csv"))
print(f"Filas: {len(df_ai):,}")
print(f"Columnas: {list(df_ai.columns)}")
df_ai.head(10)

In [ ]:
df_ai.to_parquet(f"{SILVER_PATH}raw_ai_companies.parquet", index=False)
print("✓ Guardado: silver/raw_ai_companies.parquet")

In [ ]:
print("=" * 50)
print("BRONZE COMPLETADO")
print("=" * 50)
print(f"  Big Tech Stocks : {len(df_big_tech):,} filas — {df_big_tech['ticker'].nunique()} empresas")
print(f"  GPU Companies   : {len(df_gpu):,} filas — {df_gpu['empresa'].nunique()} empresas")
print(f"  AI Companies    : {len(df_ai):,} filas")
print("\nArchivos en /silver listos para capa Silver →")